# 02 – Export Student Embeddings (Local)

Run this notebook **after downloading the student checkpoint from Kaggle**.
The 1D-CNN runs inference over all payload rows and writes `student_embeddings.npy`.

**Prerequisites**
- `payload_256.npy` – from `01_extract_payload_from_pcap.ipynb`
- `student_cnn_best.pt` – checkpoint downloaded from Kaggle notebook
  `../kaggle/01_distill_student_cnn.ipynb` (inside `student_results.zip`)

**Output** → `data/processed/student_embeddings.npy`  ← `(N_packets, D)` float32

**Next step** → `03_build_three_tier_graph.ipynb`

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PAYLOAD_NPY        = "data/interim/payload_dataset/payload_256.npy"
# Path to the student checkpoint downloaded from Kaggle.
STUDENT_CHECKPOINT = "models/student_cnn_best.pt"
OUTPUT_PATH        = "data/processed/student_embeddings.npy"

# ── Inference settings ─────────────────────────────────────────────────────────
BATCH_SIZE    = 2048
DROPOUT       = 0.1
L2_NORMALIZE  = True    # keep True for cosine similarity in graph building
FP16_OUTPUT   = False   # set True to halve disk usage (float16 vs float32)
DEVICE        = "auto"  # auto | cpu | cuda | cuda:0

In [ ]:
from pathlib import Path

payload_path = Path(PAYLOAD_NPY)
ckpt_path    = Path(STUDENT_CHECKPOINT)

assert payload_path.exists(), f"payload_256.npy not found: {payload_path}"
assert ckpt_path.exists(), (
    f"Student checkpoint not found: {ckpt_path}\n"
    "Download student_results.zip from Kaggle and extract student_cnn_best.pt to models/"
)

import numpy as np
arr = np.load(payload_path, mmap_mode="r")
print(f"payload_256.npy  : {arr.shape}  dtype={arr.dtype}")
print(f"student_cnn_best.pt: {ckpt_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
import subprocess, sys
from pathlib import Path

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m",
    "graphslm_ids.offline_path.training.export_student_embeddings",
    "--payload-npy",   PAYLOAD_NPY,
    "--checkpoint",    STUDENT_CHECKPOINT,
    "--output-path",   OUTPUT_PATH,
    "--batch-size",    str(BATCH_SIZE),
    "--dropout",       str(DROPOUT),
    "--device",        DEVICE,
]

if L2_NORMALIZE:
    cmd += ["--l2-normalize"]
else:
    cmd += ["--no-l2-normalize"]

if FP16_OUTPUT:
    cmd += ["--fp16-output"]

print("$", " ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Verify output.
import numpy as np
from pathlib import Path

emb = np.load(OUTPUT_PATH, mmap_mode="r")
out_path = Path(OUTPUT_PATH)
print(f"student_embeddings.npy: {emb.shape}  dtype={emb.dtype}")
print(f"File size             : {out_path.stat().st_size / 1e9:.2f} GB")

# Quick sanity check: L2 norms should be ~1.0 when L2_NORMALIZE=True
sample = emb[:1000].astype('float32')
norms = (sample ** 2).sum(axis=1) ** 0.5
print(f"L2 norms (first 1000 rows): mean={norms.mean():.4f}  std={norms.std():.4f}")